# Technologies.csv Generation — Norte Amazónica 2025

Generates one `Technologies.csv` per cluster (C1–C5) for EnergyScope ESMC.
Regional file carries capacity parameters only (`c_p`, `fmin_perc`, `fmax_perc`, `f_min`, `f_max`); cost parameters stay in `00_INDEP/Technologies.csv`.
Base: SA-PA reference file (Pablo Jimenez Zabalaga, 2025). Only values documented below are overwritten.

In [2]:
import os
import unicodedata
import pandas as pd

OUT_DIR = "output_energyscope"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# SA-PA reference file — Pablo Jimenez Zabalaga, 2025
base_df = pd.read_csv("data/Technologies.csv", sep=";")
base_df["Technologies param"] = base_df["Technologies param"].str.strip()

# Total households per municipality — Bolivia Census 2024
pop_df = pd.read_csv("../exctraction of data/output/CSV_final.csv")
HH_COL = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Total"

def normalize(s):
    """Strip accents + lowercase + spaces→underscore, for name matching."""
    s = unicodedata.normalize("NFD", str(s))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").lower().replace(" ", "_")

hh_lookup = {
    normalize(row["MUNICIPIO/TIOC"]): row[HH_COL]
    for _, row in pop_df.dropna(subset=["MUNICIPIO/TIOC", HH_COL]).iterrows()
}

cluster_pop = {}
for k, munis in CLUSTERS.items():
    cluster_pop[k] = sum(hh_lookup.get(normalize(m), 0) for m in munis)

total_pop = sum(cluster_pop.values())

for k, v in cluster_pop.items():
    print(f"C{k}: {v:,.0f} hh")
print(f"Total: {total_pop:,.0f} hh")

C1: 8,178 hh
C2: 802 hh
C3: 40,298 hh
C4: 15,752 hh
C5: 15,564 hh
Total: 80,594 hh


## 1. GENSET_DIESEL

`f_min` = existing installed diesel capacity (cannot be decommissioned); `f_max = 1e15`.
Values from AETN Bolivia Anuario Estadístico 2024 (*Potencia Instalada Térmica en Sistemas Aislados*).
Municipalities without AETN data (non-electrified or below threshold) get `f_min = 0`.
C1 is SIN-connected via ENDE DELBENI — no local diesel generation.

In [3]:
# Source: AETN Bolivia Anuario Estadístico 2024 — Potencia Instalada Térmica
GENSET_AETN_MW = {
    "Riberalta":              28.00,  # ENDE isolated system
    "Guayaramerín":           30.27,  # ENDE DELBENI "Guayaramerín Unificado"
    "Puerto_Gonzalo_Moreno":   4.10,  # AETN 2024
    "Sena":                    6.37,  # "El Sena" AETN 2024
    "Villa_Nueva":             2.90,  # AETN 2024
    "Cobija":                 34.20,  # ENDE isolated system
    # SIN-connected → no local diesel generation
    "Exaltación":              0.00,  # SIN via San Ignacio de Moxos
    "Reyes":                   0.00,  # SIN (ENDE DELBENI)
    "Santa_Rosa_Beni":         0.00,  # SIN (ENDE DELBENI)
    "Ixiamas":                 0.00,  # SIN-connected, no AETN isolated data
    # No AETN data → non-electrified or below AETN threshold
    "Bolpebra": 0.0, "Bella_Flor": 0.0, "Filadelfia": 0.0, "Ingavi": 0.0,
    "Nueva_Esperanza": 0.0, "Porvenir": 0.0, "Puerto_Rico": 0.0,
    "San_Lorenzo": 0.0, "San_Pedro": 0.0, "Santa_Rosa_Pando": 0.0,
    "Santos_Mercado": 0.0,
}

# Sum per cluster in GW
genset_fmin_gw = {}
for k, munis in CLUSTERS.items():
    genset_fmin_gw[k] = sum(GENSET_AETN_MW.get(m, 0.0) for m in munis) / 1000.0
    print(f"C{k}: {genset_fmin_gw[k]:.5f} GW")

C1: 0.00000 GW
C2: 0.00000 GW
C3: 0.06237 GW
C4: 0.00927 GW
C5: 0.03420 GW


## 2. Cooking stoves

`f_min` is derived from current cooking practices (Census 2024) so the optimizer cannot remove existing stove capacity:

$$f_{\min} = \frac{n_{\text{hh}} \times FE}{\text{COEFF} \times 10^6 \times c_p \times 8760}$$

Energy intensities from a rural Bolivia field study (Oriente/Yungas zones). EnergyScope coefficients from `Layers_in_out.csv`.

In [4]:
# Source: Bolivia Census 2024, CSV_final — (hh_wood, hh_lpg, hh_elec)
COOKING_DATA = {
    "Exaltación":            (1090, 332,   1),
    "Reyes":                 (1482, 1848,  4),
    "Santa_Rosa_Beni":       (917,  1764,  6),
    "Ixiamas":               (1528, 1679,  1),
    "Bolpebra":              (340,  452,   1),
    "Guayaramerín":          (1575, 8506, 19),
    "Riberalta":             (4512, 20740, 47),
    "Puerto_Gonzalo_Moreno": (927,  991,   3),
    "Bella_Flor":            (409,  803,   2),
    "Filadelfia":            (613,  1857,  3),
    "Ingavi":                (350,  293,   0),
    "Nueva_Esperanza":       (190,  293,   2),
    "Porvenir":              (223,  1950,  2),
    "Puerto_Rico":           (405,  1513,  6),
    "San_Lorenzo":           (763,  1050,  1),
    "San_Pedro":             (520,  85,    0),
    "Santa_Rosa_Pando":      (235,  615,   2),
    "Santos_Mercado":        (359,  238,   0),
    "Sena":                  (1312, 1487,  2),
    "Villa_Nueva":           (396,  302,   1),
    "Cobija":                (300,  13727, 70),
}

FE_WOOD = 6760  # kWh/hh/yr — Source: Perplexity rural Bolivia field study (Oriente/Yungas)
FE_LPG  = 6970  # kWh/hh/yr — same source

COEFF_WOOD = 6.25      # Source: Layers_in_out.csv — STOVE_WOOD, COOKING column
COEFF_LPG  = 1.559258  # Source: Layers_in_out.csv — STOVE_LPG,  COOKING column
CP_STOVE   = 0.1875    # c_p for all stoves (SA-PA base)

stove_wood_fmin = {}
stove_lpg_fmin  = {}

for k, munis in CLUSTERS.items():
    wood_gwh = 0
    lpg_gwh  = 0
    for m in munis:
        hh_wood, hh_lpg, _ = COOKING_DATA[m]
        wood_gwh += hh_wood * FE_WOOD / COEFF_WOOD / 1e6
        lpg_gwh  += hh_lpg  * FE_LPG  / COEFF_LPG  / 1e6
    stove_wood_fmin[k] = wood_gwh / (CP_STOVE * 8760)
    stove_lpg_fmin[k]  = lpg_gwh  / (CP_STOVE * 8760)

print(f"{'':8} {'STOVE_WOOD':>12} {'STOVE_LPG':>12}")
for k in range(1, 6):
    print(f"C{k}       {stove_wood_fmin[k]:12.7f} {stove_lpg_fmin[k]:12.7f}")

           STOVE_WOOD    STOVE_LPG
C1          0.0033037    0.0153030
C2          0.0002239    0.0012301
C3          0.0046188    0.0822902
C4          0.0038029    0.0285377
C5          0.0001976    0.0373581


## 3. PV_UTILITY and population-scaled technologies

In [ ]:
# Source: AETN 2024 — Cobija only: 5.10 MW (ENDE GUARACACHI S.A.)
PV_FMIN_GW = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.00510}

# Technologies whose SA-PA f_min is scaled by the cluster's share of total households.
# Not in this list: GENSET_DIESEL, PV_UTILITY, STOVE_* (computed above),
# REGASIFICATION, LNG_STORAGE (see section 4).
# Infrastructure (HVAC_LINE, GAS_PIPELINE …) and LED fmin_perc are left unchanged.
POPULATION_SCALED = [
    "DIESEL_STORAGE", "GASOLINE_STORAGE", "LPG_STORAGE",
    "JET_FUEL_STORAGE",
    "CAR_GASOLINE_PRIVATE", "SUV_GASOLINE_PRIVATE",
    "PICKUP_TRUCK_GASOLINE_PRIVATE", "MOTORCYCLE_GASOLINE_PRIVATE",
    "BUS_GASOLINE_PUBLIC", "CAR_DIESEL_PRIVATE", "SUV_DIESEL_PRIVATE",
    "PICKUP_TRUCK_DIESEL_PRIVATE", "BUS_DIESEL_PUBLIC",
    "TRUCK_DIESEL_P", "VAN_DIESEL", "TRUCK_GASOLINE", "VAN_GASOLINE",
    "FISH_MACHINERY_DIESEL", "FISH_MACHINERY_EL",
    "IND_MACHINERY_EL", "COMM_MACHINERY_DIESEL", "COMM_MACHINERY_EL",
    "AGR_MACHINERY_DIESEL", "AGR_MACHINERY_EL",
    "DEC_DIRECT_ELEC", "DEC_BOILER_GAS",
    "IND_BOILER_WOOD", "IND_BOILER_OIL", "IND_BOILER_DIESEL",
    "STOVE_NG", "STOVE_OIL",
]

# Pre-read SA-PA f_min for each technology before the cluster loop
base_fmin = {}
for tech in POPULATION_SCALED:
    row = base_df.loc[base_df["Technologies param"] == tech, "f_min"]
    base_fmin[tech] = float(row.values[0]) if len(row) else 0.0

## 4. REGASIFICATION and LNG_STORAGE

`f_min = 0` for both in all clusters — not population-scaled.
AETN 2024 confirms no LNG regasification stations in Norte Amazónica
(Riberalta, Guayaramerín, Cobija all use diesel only for generation).

## 5. SHS rows and output generation

Three Solar Home System technologies are appended to every cluster.
Cost parameters are in `00_INDEP/Technologies.csv` (Roger Arias thesis); `share_dispersion = 0` is set in `Misc.json`.

In [ ]:
shs_rows = [
    # tech          c_p   fmin_perc  fmax_perc  f_min  f_max
    ("PV_HS",       1.00, 0, 1, 0, 1e15),
    ("HS_DIESEL",   0.85, 0, 1, 0, 1e15),
    ("BATT_HS",     1.00, 0, 1, 0, 1e15),
]
shs_df = pd.DataFrame(shs_rows, columns=base_df.columns)

for k, munis in CLUSTERS.items():
    df = base_df.copy()
    share = cluster_pop[k] / total_pop

    df.loc[df["Technologies param"] == "GENSET_DIESEL", "f_min"] = genset_fmin_gw[k]
    df.loc[df["Technologies param"] == "GENSET_DIESEL", "f_max"] = 1e15

    df.loc[df["Technologies param"] == "STOVE_WOOD", "f_min"] = stove_wood_fmin[k]
    df.loc[df["Technologies param"] == "STOVE_LPG",  "f_min"] = stove_lpg_fmin[k]
    df.loc[df["Technologies param"] == "STOVE_ELEC", "f_min"] = 0.0

    df.loc[df["Technologies param"] == "PV_UTILITY", "f_min"] = PV_FMIN_GW[k]

    for tech in POPULATION_SCALED:
        mask = df["Technologies param"] == tech
        if mask.any():
            df.loc[mask, "f_min"] = base_fmin[tech] * share

    # REGASIFICATION and LNG_STORAGE f_min = 0:
    # AETN 2024 confirms no LNG regasification stations in Norte Amazónica
    # (Riberalta, Guayaramerín, Cobija all use diesel only for generation)
    df.loc[df["Technologies param"] == "REGASIFICATION", "f_min"] = 0.0
    df.loc[df["Technologies param"] == "LNG_STORAGE",    "f_min"] = 0.0

    # Remove any pre-existing SHS rows, then append
    df = df[~df["Technologies param"].isin(shs_df["Technologies param"])]
    df = pd.concat([df, shs_df], ignore_index=True)

    out_path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_csv(out_path, sep=";", index=False)
print(f"Saved Technologies.csv files")

## 6. Verification

In [ ]:
clusters_out = {}
for k in range(1, 6):
    path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    clusters_out[k] = pd.read_csv(path, sep=";")

def lookup(df, tech, col):
    row = df.loc[df["Technologies param"] == tech, col]
    return float(row.values[0]) if len(row) else float("nan")

checks = [
    ("GENSET_DIESEL",  "f_min"),
    ("STOVE_WOOD",     "f_min"),
    ("STOVE_LPG",      "f_min"),
    ("PV_UTILITY",     "f_min"),
    ("DIESEL_STORAGE", "f_min"),
    ("LNG_STORAGE",    "f_min"),
    ("REGASIFICATION", "f_min"),
]

header = f"{'Technology':<30}" + "".join(f"  C{k:>9}" for k in range(1, 6))
print(header)
print("-" * len(header))
for tech, col in checks:
    vals = [lookup(clusters_out[k], tech, col) for k in range(1, 6)]
    print(f"{tech+' '+col:<30}" + "".join(f"  {v:>9.5f}" for v in vals))

print()
for shs in ["PV_HS", "HS_DIESEL", "BATT_HS"]:
    present = ["✓" if shs in clusters_out[k]["Technologies param"].values else "✗" for k in range(1, 6)]
    print(f"{shs+' present':<30}" + "".join(f"  {p:>9}" for p in present))